In [3]:
# ── 1. Model Initialization & Target Selection ─────────────────────────────
import pandas as pd
import numpy as np
import scipy.stats as stats
from sklearn.linear_model import ElasticNet
from sklearn.preprocessing import MinMaxScaler
from itertools import product
import warnings

warnings.filterwarnings('ignore')

# ==========================================
# CONTROL PANEL: CHOOSE YOUR TARGET
# ==========================================
# Change this single variable to run the entire ML pipeline on a different target!
# Options: 'HFRXGL', 'Monster_Index', 'MXWO', 'MXWD', 'LEGATRUU'

MAIN_TARGET = 'Monster_Index'

# ==========================================

# --- 1. LOAD RAW DATA ---
# Replace with your actual file path if it's in a different folder
file_path = 'Dataset3_PortfolioReplicaStrategy.xlsx'

print(f" Loading raw data from: {file_path}...")
# Assuming the first column contains the dates
data = pd.read_excel(file_path, index_col=0, parse_dates=True)

# Clean up column names just in case there are hidden spaces
data.columns = data.columns.str.strip()


# --- 2. DEFINE UNIVERSE & CALCULATE RETURNS ---
FUTURES = ['RX1', 'TY1', 'GC1', 'CO1', 'ES1', 'VG1', 'NQ1', 'LLL1', 'TP1', 'DU1', 'TU2']
BASE_INDICES = ['HFRXGL', 'MXWO', 'MXWD', 'LEGATRUU']

# Ensure we only calculate returns for columns that actually exist in the file
available_cols = [col for col in BASE_INDICES + FUTURES if col in data.columns]
all_returns = data[available_cols].pct_change().dropna()

# --- 3. CONSTRUCT THE "MONSTER INDEX" ---
monster_weights = {
    'HFRXGL': 0.40,
    'MXWO': 0.20,
    'MXWD': 0.20,
    'LEGATRUU': 0.20
}
# Multiply base indices by weights and sum to create the new target
all_returns['Monster_Index'] = sum(all_returns[comp] * w for comp, w in monster_weights.items() if comp in all_returns.columns)

print(f" INITIALIZING PIPELINE: Target set to [{MAIN_TARGET}]")

# --- 4. EXTRACT ALIGNED ARRAYS ---
y = all_returns[MAIN_TARGET]
X = all_returns[FUTURES]

# Convert to fast numpy arrays for the rolling ML loop
X_values = X.values
y_values = y.values
dates_array = X.index.to_numpy()

# --- 5. HYPERPARAMETER GRID ---
l1_ratios = [0.0, 0.2, 0.4, 0.6, 0.8, 1.0]
rolling_windows = [52, 104, 156]  # 1Y, 2Y, 3Y
alphas = [0.0001, 0.001, 0.01]

# --- 6. RISK PARAMETERS & VaR ENGINE ---
var_confidence = 0.01
var_horizon = 4
max_var_threshold = 0.08

def calculate_var(returns, confidence=0.01, horizon=4, method='historical'):
    """Calculates Value at Risk (VaR) handling empirical fat tails."""
    if len(returns) == 0:
        return np.nan

    if method == 'historical':
        weekly_var = -np.percentile(returns, confidence * 100)
    elif method == 'gaussian':
        weekly_var = -stats.norm.ppf(confidence) * np.std(returns)
    else:
        raise ValueError("Method must be 'historical' or 'gaussian'")

    return weekly_var * np.sqrt(horizon)

print("Data loaded, aligned, and ready for Machine Learning.")

 Loading raw data from: Dataset3_PortfolioReplicaStrategy.xlsx...
 INITIALIZING PIPELINE: Target set to [Monster_Index]
Data loaded, aligned, and ready for Machine Learning.
